In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path

# Read the file


In [ ]:
"""
Input is the energy-performance file, in this notebook the energy efficiency metrics will be added
A configuration is a set of: model, batch_size, dropout rate, learning rate that was run for 10 epochs.

Configurations used:
"batch_size": 128, 256, 64, 96, 156, 72
"lr": 0.03, 0.035, 0.04, 0.025, 0.02, 0.01
"dropout": 0.025, 0.075, 0.01 , 0.0, 0.1, 0.05 
"epochs": 10
"optimizer": SGD
"model": "ViT-B/32", "ViT-S/32", "ViT-T/16", "ViT-S/16", "ViT-B/16", "ViT-L/32"

What other columns hold:
"unique_id": model concatinated with exp_id
"exp_id": id for a configuration that is run 
"duration": seconds
"total_energy_J" CodeCarbon energy converted to Joule
"accuracy_0": validation accuracy at epoch 1
"accuracy": validation accuracy at epoch 30
"precision_0": validation precision at epoch 1
"precision": validation accuracy at epoch 30
"recall_0": validation recall at epoch 1
"recall": validation recall at epoch 30
"specificity_0": validation specificity at epoch 1
"specificity": validation specificity at epoch 30
"test_accuracy": test accuracy
"Pg_0": Pg at epoch 1
"Pg": Pg at epoch 30
"n_samples": samples processed in one epoch
"rate": train/val split. Default is 0.8
"tot_samples": total samples processed over all epochs
"

"""



In [ ]:
# Read file
data = pd.read_csv(r"..\Datasets\For analysis\Before analysis\CIFAR100_energy_performance.csv")
data.columns

# Define energy efficiency metrics


In [14]:
# Defining and adding energy efficiency metrics
# The metrics are multiplied with nr_epochs so it is easier to plot 

# Configurations
n_epochs = 10
J_per_epoch = data["total_energy_J"] / n_epochs
duration_per_epoch = data["duration"] / n_epochs

# Read FLOPS data
FLOPS = pd.read_json("FLOPS_v2.json")
flops_series = FLOPS.loc["FLOPS_fvcore"]

# Calculate SPJ
data["tot_samples"] = n_epochs*data["n_samples"]
data["SPJ"] = data["tot_samples"] / data["total_energy_J"]

# EDP (Energy-Delay Product) in Joule-seconds
data["EDP"] = data["total_energy_J"] * data["duration"]
data["EDPinv"] = 1/data["EDP"]

# EDP per epoch
data["EDPinv_per_epoch"] = data["EDPinv"] / data["epochs"]

# FLOPS
data["FLOPS"] = data["model"].map(flops_series)
data["FLOPS_"] = data["FLOPS"]*n_epochs
data["FPJ"] = data["FLOPS_"]/ data["total_energy_J"]

data["Eg"] = data["FPJ"] * data["EDPinv"] * data["SPJ"]


# Remove some runs that failed
data = data[~((data["model"] == "vit_b_32") & (data["Eg"] < 0.002))]


In [16]:
# Save
data.to_csv(r"..\Datasets\For analysis\full_data_CIFAR100_v3.csv", index=False)